In [1]:
# Imports used by preprocessing and the recommendation model.
import ast
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# Load the source data and keep products with both feature sources available.
DATA_DIR = Path("data")
df = pd.read_csv(DATA_DIR / "product_info.csv")

products = (
    df.dropna(subset=["highlights", "ingredients"])
    .copy()
    .reset_index(drop=True)
)

In [ ]:
# convert str of list into python list
def parse_highlights(value):
    
    
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return [str(value)]


products["highlights_list"] = products["highlights"].apply(
    parse_highlights
)

In [4]:
def clean_highlight(text):
    # convert to lowercase
    text = str(text).lower().strip()

    # normalize unicode characters
    text = unicodedata.normalize("NFKC", text)

    # replace hyphens with spaces
    text = re.sub(r"[-‐‑–—]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text

products["cleaned_highlights"] = (
    products["highlights_list"].apply(
        lambda items: [clean_highlight(h) for h in items]
    )
)

In [5]:
#  now splite "good for: A,B,C" into "good for A, good for B ..."
skin_type_mapping = {
    "best for dry skin": ["skin_dry"],
    "best for oily skin": ["skin_oily"],
    "best for normal skin": ["skin_normal"],
    "best for combination skin": ["skin_combination"],
    "best for dry, combo, normal skin": [
        "skin_dry", "skin_combination", "skin_normal"
    ],
    "best for oily, combo, normal skin": [
        "skin_oily", "skin_combination", "skin_normal"
    ]
}

#  revomoving hihglights non-related
marketing_highlights = {
    "community favorite",
    "clean at sephora",
    "clean + planet positive",
    "black owned at sephora"
}

def normalize_highlight(highlight):
    # Normalize text formatting
    h = highlight.lower().strip()
    h = h.replace("-", " ")
    h = re.sub(r"\s+", " ", h)

    # Remove marketing features from similarity
    if h in marketing_highlights or h.startswith("allure "):
        return []

    # Expand combined skin-type labels
    if h in skin_type_mapping:
        return skin_type_mapping[h]

    # Normalize remaining highlights
    return [h.replace(" ", "_")]


    
def preprocess_highlights(highlights):
    normalized = []

    for highlight in highlights:
        normalized.extend(normalize_highlight(highlight))

    # Remove duplicates while preserving order
    return list(dict.fromkeys(normalized))


products["normalized_highlights"] = (
    products["cleaned_highlights"].apply(preprocess_highlights)
)

In [ ]:
def normalize_ingredient(name):
    
    # clean some formatiing 
    if not isinstance(name, str):
        return ""

    #  decode some html values
    name = html.unescape(name)
    name = unicodedata.normalize("NFKC", name).casefold()

    # replace all possible dashes with one type
    name = re.sub(r"[‐‑‒–—−]", "-", name)

    # remove extra white spaces
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s*/\s*", "/", name)

    name = re.sub(r"[\s*†‡]+$", "", name)
    # removes speces, periods, semicolms
    return name.strip(" .;")

# splite the ingridients, while being mindful of the brakets in notation
# "Water, Example Extract (leaf, stem), Glycerin" -> ["water", "example extract (leaf, stem)", "glycerin"]
def split_ingredient_text(text):
    #  to store the names of ingridients
    ingredients = []
    
    # the current chars of the ingridient
    current = []
    
    # opened parathethis 
    stack = []

    for char in text:
        if char in "([":
            stack.append(char)

        elif char in ")]":
            expected = "(" if char == ")" else "["
            if not stack or stack[-1] != expected:
                raise ValueError("unbalanced ingredient parentheses/brackets")
            stack.pop()

        if char == "," and not stack:
            ingredient = normalize_ingredient("".join(current))
            if ingredient:
                ingredients.append(ingredient)
            current = []
        else:
            current.append(char)

    if stack:
        raise ValueError("Unbalanced ingredient parentheses/brackets")

    ingredient = normalize_ingredient("".join(current))
    if ingredient:
        ingredients.append(ingredient)

    return ingredients


# also assignes a stustues to an ingridient list
# missing -> no values
# invalid-list - > invalid python list
# invalid section - contains elem that is not a sring
# needs_section_review - contains more than one section of ingridients, like seperate part ususlly devided by : 
# needs_section_review - missing brakets
# empty
def parse_ingredients(value):
    if isinstance(value, list):
        sections = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return {"ingredients": [], "status": "missing"}

        if value.startswith("["):
            try:
                # converts text into actual list python
                sections = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return {"ingredients": [], "status": "invalid_list"}
        else:
            sections = [value]

    else:
        return {"ingredients": [], "status": "missing"}

    if not isinstance(sections, list):
        return {"ingredients": [], "status": "invalid_list"}

    if not all(isinstance(section, str) for section in sections):
        return {"ingredients": [], "status": "invalid_section"}

    sections = [section.strip() for section in sections if section.strip()]

    if not sections:
        return {"ingredients": [], "status": "missing"}

    if len(sections) > 1:
        return {"ingredients": [], "status": "needs_section_review"}

    try:
        ingredients = split_ingredient_text(sections[0])
    except ValueError:
        return {"ingredients": [], "status": "needs_delimiter_review"}

    return {
        "ingredients": ingredients,
        "status": "parsed" if ingredients else "empty",
    }

In [7]:
APPROVED_CORRECTIONS = {
    "hydroxyacetophe-none": "hydroxyacetophenone",
    "pan-thenol": "panthenol",
}

APPROVED_ALIASES = {
    "aqua/water/eau": "water",
    "water/aqua/eau": "water",
    "aqua (water)": "water",
    "water (aqua)": "water",
}

CANONICAL_MAP = {
    normalize_ingredient(source): normalize_ingredient(target)
    for source, target in {
        **APPROVED_CORRECTIONS,
        **APPROVED_ALIASES,
    }.items()
}


def canonicalize_ingredients(ingredients):
    # applying the approved replacement
    cleaned = []

    for ingredient in ingredients:
        name = normalize_ingredient(ingredient)
        name = CANONICAL_MAP.get(name, name)

        if name:
            cleaned.append(name)

    return list(dict.fromkeys(cleaned))

In [8]:
parsed = pd.DataFrame(
    products["ingredients"].apply(parse_ingredients).tolist(),
    index=products.index,
)

products["ingredients_list"] = parsed["ingredients"]
products["ingredient_parse_status"] = parsed["status"]


products["normalized_ingredients"] = (
    products["ingredients_list"].apply(canonicalize_ingredients)
)

print(products["ingredient_parse_status"].value_counts())

# recreate skincare after updating products.
skincare = products.loc[
    products["primary_category"].eq("Skincare")
].copy()

<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.


ingredient_parse_status
parsed                    4987
needs_section_review       798
needs_delimiter_review      35
Name: count, dtype: int64


In [ ]:
# Keep individual skincare formulas.
eligible_categories = {
    "Cleansers",
    "Eye Care",
    "Lip Balms & Treatments",
    "Masks",
    "Moisturizers",
    "Self Tanners",
    "Sunscreen",
    "Treatments",
}


# working only with skincare from chosen ligible_categories and parsed, so dont deal with complicated stuff
skincare_model = products.loc[
    products["primary_category"].eq("Skincare")
    & products["secondary_category"].isin(eligible_categories)
    & products["ingredient_parse_status"].eq("parsed")
    & products["normalized_ingredients"].map(bool)
].copy()

skincare_model["price_usd"] = pd.to_numeric(
    skincare_model["price_usd"],
    errors="coerce",
)

skincare_model = (
    skincare_model
    .drop_duplicates(subset="product_id")
    .reset_index(drop=True)
)

if skincare_model.empty:
    raise ValueError("No eligible products. Check the preprocessing cells.")

print("Products available:", len(skincare_model))

Products available: 1525


In [ ]:
# make rows aligned with both matrices.
skincare_model = skincare_model.reset_index(drop=True)

# initialised the Tf-idf using ingridients
ingredient_vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

# actully apply the vectorised and assign frequencies to the ingridients
X_ingredients = ingredient_vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

# each highlight becomes a binary feature: present or absent
highlight_encoder = MultiLabelBinarizer(sparse_output=True)

X_highlights = highlight_encoder.fit_transform(
    skincare_model["normalized_highlights"]
)


# makes a dictionary products id: row number
product_positions = {
    product_id: position
    for position, product_id in enumerate(skincare_model["product_id"])
}

print("Ingredient feature matrix:", X_ingredients.shape)
print("Highlight feature matrix:", X_highlights.shape)

Ingredient feature matrix: (1525, 5156)
Highlight feature matrix: (1525, 67)


In [13]:
def recommend_dupes(
    product_id,
    top_k=5,
    # given the ingridients the weight of 70% theofore the highlights are weighted at 30%
    ingredient_weight=0.7,
    different_brand=False,
):
    if product_id not in product_positions:
        raise ValueError("Product ID is not in skincare_model.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k must be a positive integer.")

    if not 0 < ingredient_weight < 1:
        raise ValueError("ingredient_weight must be between 0 and 1.")

    position = product_positions[product_id]
    
    # what are we looking an alternative for
    query = skincare_model.iloc[position]

    if pd.isna(query["price_usd"]) or query["price_usd"] <= 0:
        raise ValueError("The selected product needs a valid price.")


    if X_highlights[position].nnz == 0:
        raise ValueError(
            "This product has no usable highlights to compare."
        )

    # calculate the two similarity scores separately.
    ingredient_scores = cosine_similarity(
        X_ingredients[position],
        X_ingredients,
    ).ravel()

    highlight_scores = cosine_similarity(
        X_highlights[position],
        X_highlights,
    ).ravel()

    # starting weights: 70% ingredients and 30% highlights by default, got these numbers out of head
    combined_scores = (
        ingredient_weight * ingredient_scores
        + (1 - ingredient_weight) * highlight_scores
    )

    # only looking at the comparable product categories
    mask = skincare_model["secondary_category"].eq(
        query["secondary_category"]
    )

    if pd.notna(query["tertiary_category"]):
        mask &= skincare_model["tertiary_category"].eq(
            query["tertiary_category"]
        )
    else:
        mask &= skincare_model["tertiary_category"].isna()

    # alternatives MUST have a lower package price
    mask &= (
        skincare_model["price_usd"].gt(0)
        & skincare_model["price_usd"].lt(query["price_usd"])
    )

    mask &= skincare_model["product_id"].ne(product_id)

    if different_brand:
        mask &= skincare_model["brand_name"].ne(query["brand_name"])

    # require some overlap in BOTH ingredients and highlights
    mask &= (ingredient_scores > 0) & (highlight_scores > 0)

    results = skincare_model.loc[mask].copy()
    selected = mask.to_numpy()

    results["ingredient_similarity"] = ingredient_scores[selected]
    results["highlight_similarity"] = highlight_scores[selected]
    results["combined_similarity"] = combined_scores[selected]

    results = results.sort_values(
        ["combined_similarity", "price_usd", "product_id"],
        ascending=[False, True, True],
    ).head(top_k).copy()

    # explain what each alternative shares with the selected product
    query_ingredients = set(query["normalized_ingredients"])
    query_highlights = set(query["normalized_highlights"])

    results["shared_ingredients"] = results[
        "normalized_ingredients"
    ].apply(lambda items: sorted(query_ingredients & set(items)))

    results["shared_highlights"] = results[
        "normalized_highlights"
    ].apply(lambda items: sorted(query_highlights & set(items)))

    results["package_savings_usd"] = (
        query["price_usd"] - results["price_usd"]
    )

    results["package_savings_pct"] = (
        100 * results["package_savings_usd"] / query["price_usd"]
    )

    columns = [
        "product_id",
        "product_name",
        "brand_name",
        "size",
        "price_usd",
        "ingredient_similarity",
        "highlight_similarity",
        "combined_similarity",
        "package_savings_usd",
        "package_savings_pct",
        "shared_highlights",
        "shared_ingredients",
    ]

    return results[columns].reset_index(drop=True)

In [ ]:
# HERE put the product id you are looking dupes for 
# for example select the first product with usable highlights and a valid price
example_products = skincare_model.loc[
    skincare_model["normalized_highlights"].map(bool)
    & skincare_model["price_usd"].gt(0)
]




if example_products.empty:
    raise ValueError("No products have both usable highlights and a valid price.")

# assign query_id to the product if you are looking a dupe for 
query_id = example_products.iloc[0]["product_id"]

display(
    skincare_model.loc[
        skincare_model["product_id"].eq(query_id),
        [
            "product_id", "product_name", "brand_name", "size",
            "price_usd", "normalized_highlights",
        ],
    ]
)

# Every returned alternative must be cheaper and share ingredients and highlights.
recommendations = recommend_dupes(
    product_id=query_id,
    top_k=5,
    ingredient_weight=0.7,
    different_brand=False,
)

display(
    recommendations.round({
        "ingredient_similarity": 3,
        "highlight_similarity": 3,
        "combined_similarity": 3,
        "package_savings_usd": 2,
        "package_savings_pct": 1,
    })
)

if recommendations.empty:
    print("No cheaper alternatives match both ingredients and highlights.")

# Savings compare package prices, not equal volumes or weights.
# Shared ingredients do not establish equal concentrations or performance.

,product_id,product_name,brand_name,size,price_usd,normalized_highlights
2,P432045,GENIUS Liquid Collagen Lip Treatment,Algenist,.5 oz / 15 mL,29.0,"[vegan, good_for:_loss_of_firmness, plumping, ..."


,product_id,product_name,brand_name,size,price_usd,ingredient_similarity,highlight_similarity,combined_similarity,package_savings_usd,package_savings_pct,shared_highlights,shared_ingredients
0,P455936,Lip Butter Balm,Summer Fridays,.5 oz/ 15 g,24.0,0.005,0.500,0.154,5.0,17.2,"[plumping, skin_combination, skin_normal, vegan]",[tocopherol]
1,P500329,Plump + Juicy Lip Booster Buttery Treatment,Kosas,0.5 oz / 15 mL,25.0,0.004,0.474,0.145,4.0,13.8,"[hypoallergenic, plumping, vegan]",[tocopherol]
2,P500744,Clean Lip Balm & Scrub,SEPHORA COLLECTION,0.12 oz / 3.5 g,7.0,0.015,0.433,0.141,22.0,75.9,"[skin_combination, skin_dry, skin_normal]","[helianthus annuus (sunflower) seed oil, tocop..."
3,P503878,Pout Preserve Peptide Lip Treatment,OLEHENRIKSEN,.4 oz / 12 mL,22.0,0.011,0.354,0.114,7.0,24.1,"[good_for:_loss_of_firmness, vegan]","[helianthus annuus (sunflower) seed oil, tocop..."
4,P500307,Cherry Treat Conditioning + Strengthening Lip Oil,Fenty Skin,.19 oz / 5.6 mL,24.0,0.068,0.177,0.101,5.0,17.2,[vegan],"[benzyl benzoate, dextrin palmitate]"
